In [4]:
# ==================================================================================================
# CELL 1 — IMPORT REQUIRED LIBRARIES
# ==================================================================================================

import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 100)
print("PHASE 03.03 — CROSS-MODAL LOSS DESIGN")
print("REQUIRED LIBRARIES IMPORTED")
print("=" * 100)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

print("\n✓ Numerical libraries ready")
print("✓ PyTorch ready")
print("✓ Loss functions ready")
print("=" * 100)

PHASE 03.03 — CROSS-MODAL LOSS DESIGN
REQUIRED LIBRARIES IMPORTED
PyTorch version: 2.13.0+cpu
CUDA available : False

✓ Numerical libraries ready
✓ PyTorch ready
✓ Loss functions ready


In [5]:
# ==================================================================================================
# CELL 2 — CONFIGURE REPRODUCIBILITY
# ==================================================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print("=" * 100)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print(f"Random seed: {RANDOM_SEED}")

print("\n✓ Python seed configured")
print("✓ NumPy seed configured")
print("✓ PyTorch seed configured")

print("=" * 100)

REPRODUCIBILITY CONFIGURATION
Random seed: 42

✓ Python seed configured
✓ NumPy seed configured
✓ PyTorch seed configured


In [7]:
# ==================================================================================================
# CELL 3 — DEFINE PROJECT ROOT
# ==================================================================================================

from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening"
)

print("=" * 100)
print("PROJECT ROOT CONFIGURATION")
print("=" * 100)

print(PROJECT_ROOT)
print(f"\nExists: {PROJECT_ROOT.exists()}")

assert PROJECT_ROOT.exists(), "PROJECT_ROOT does not exist."

print("\n✓ PROJECT ROOT VERIFIED")
print("=" * 100)

PROJECT ROOT CONFIGURATION
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

Exists: True

✓ PROJECT ROOT VERIFIED


In [8]:
# ==================================================================================================
# CELL 4 — DEFINE REPRESENTATION-AWARE DATASET PATHS
# ==================================================================================================

PAIR_DATA_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "02_cross_modal_pairing" /
    "pairs" /
    "representation_aware_split"
)

TRAIN_PAIR_PATH = (
    PAIR_DATA_DIR /
    "02_08_train_representation_disjoint_pairs.npz"
)

VALIDATION_PAIR_PATH = (
    PAIR_DATA_DIR /
    "02_08_validation_representation_disjoint_pairs.npz"
)

print("=" * 100)
print("DATASET PATH CONFIGURATION")
print("=" * 100)

print("\nTraining dataset:")
print(TRAIN_PAIR_PATH)

print("\nValidation dataset:")
print(VALIDATION_PAIR_PATH)

assert TRAIN_PAIR_PATH.exists(), "Training dataset not found."
assert VALIDATION_PAIR_PATH.exists(), "Validation dataset not found."

print("\n✓ TRAINING DATASET FOUND")
print("✓ VALIDATION DATASET FOUND")
print("=" * 100)

DATASET PATH CONFIGURATION

Training dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

Validation dataset:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

✓ TRAINING DATASET FOUND
✓ VALIDATION DATASET FOUND


In [9]:
# ==================================================================================================
# CELL 5 — LOAD REPRESENTATION-AWARE CROSS-MODAL PAIRS
# ==================================================================================================

print("=" * 100)
print("LOADING CROSS-MODAL PAIR DATASETS")
print("=" * 100)

train_data = np.load(TRAIN_PAIR_PATH)
validation_data = np.load(VALIDATION_PAIR_PATH)

X_source_train = train_data["source_embeddings"]
X_target_train = train_data["target_embeddings"]
y_train = train_data["pair_labels"]

X_source_val = validation_data["source_embeddings"]
X_target_val = validation_data["target_embeddings"]
y_val = validation_data["pair_labels"]

print("\nTRAINING DATASET")
print("-" * 100)
print(f"Source embeddings : {X_source_train.shape}")
print(f"Target embeddings : {X_target_train.shape}")
print(f"Pair labels       : {y_train.shape}")

print("\nVALIDATION DATASET")
print("-" * 100)
print(f"Source embeddings : {X_source_val.shape}")
print(f"Target embeddings : {X_target_val.shape}")
print(f"Pair labels       : {y_val.shape}")

print("\n" + "=" * 100)
print("✓ DATASETS LOADED SUCCESSFULLY")
print("=" * 100)

LOADING CROSS-MODAL PAIR DATASETS

TRAINING DATASET
----------------------------------------------------------------------------------------------------
Source embeddings : (1776, 32)
Target embeddings : (1776, 32)
Pair labels       : (1776,)

VALIDATION DATASET
----------------------------------------------------------------------------------------------------
Source embeddings : (448, 32)
Target embeddings : (448, 32)
Pair labels       : (448,)

✓ DATASETS LOADED SUCCESSFULLY


In [10]:
# ==================================================================================================
# CELL 6 — VERIFY BINARY CROSS-MODAL RELATIONSHIP LABELS
# ==================================================================================================

print("=" * 100)
print("VERIFYING CROSS-MODAL RELATIONSHIP LABELS")
print("=" * 100)

train_unique = np.unique(y_train)
val_unique = np.unique(y_val)

print("\nTraining unique labels :", train_unique)
print("Validation unique labels:", val_unique)

assert set(train_unique).issubset({0, 1})
assert set(val_unique).issubset({0, 1})

print("\nTRAINING DISTRIBUTION")
print("-" * 100)
print(f"Positive pairs (1): {np.sum(y_train == 1)}")
print(f"Negative pairs (0): {np.sum(y_train == 0)}")

print("\nVALIDATION DISTRIBUTION")
print("-" * 100)
print(f"Positive pairs (1): {np.sum(y_val == 1)}")
print(f"Negative pairs (0): {np.sum(y_val == 0)}")

print("\n" + "=" * 100)
print("✓ BINARY RELATIONSHIP LABELS VERIFIED")
print("✓ SUITABLE FOR BINARY CROSS-ENTROPY LOSS")
print("=" * 100)

VERIFYING CROSS-MODAL RELATIONSHIP LABELS

Training unique labels : [0 1]
Validation unique labels: [0 1]

TRAINING DISTRIBUTION
----------------------------------------------------------------------------------------------------
Positive pairs (1): 888
Negative pairs (0): 888

VALIDATION DISTRIBUTION
----------------------------------------------------------------------------------------------------
Positive pairs (1): 224
Negative pairs (0): 224

✓ BINARY RELATIONSHIP LABELS VERIFIED
✓ SUITABLE FOR BINARY CROSS-ENTROPY LOSS


In [11]:
# ==================================================================================================
# CELL 7 — DEFINE CROSS-MODAL LEARNING OBJECTIVE
# ==================================================================================================

print("=" * 100)
print("CROSS-MODAL LEARNING OBJECTIVE")
print("=" * 100)

print("""
For each cross-modal pair:

    Source Representation = S
    Target Representation = T

The baseline model produces:

    z = f(S, T)

where:

    z = raw relationship logit

The sigmoid function converts the logit into probability:

                   1
    p = -------------------------
              1 + exp(-z)

where:

    p = probability that the pair represents a positive relationship

Learning objective:

    Positive Pair (y = 1)
        → probability should approach 1

    Negative Pair (y = 0)
        → probability should approach 0

Therefore, the model must minimize the difference between:

    Predicted relationship probability

and

    True pair relationship label
""")

print("=" * 100)

CROSS-MODAL LEARNING OBJECTIVE

For each cross-modal pair:

    Source Representation = S
    Target Representation = T

The baseline model produces:

    z = f(S, T)

where:

    z = raw relationship logit

The sigmoid function converts the logit into probability:

                   1
    p = -------------------------
              1 + exp(-z)

where:

    p = probability that the pair represents a positive relationship

Learning objective:

    Positive Pair (y = 1)
        → probability should approach 1

    Negative Pair (y = 0)
        → probability should approach 0

Therefore, the model must minimize the difference between:

    Predicted relationship probability

and

    True pair relationship label



In [12]:
# ==================================================================================================
# CELL 8 — DEMONSTRATE LOGIT TO PROBABILITY CONVERSION
# ==================================================================================================

print("=" * 100)
print("LOGIT TO PROBABILITY CONVERSION")
print("=" * 100)

example_logits = torch.tensor(
    [-5.0, -2.0, 0.0, 2.0, 5.0]
)

example_probabilities = torch.sigmoid(example_logits)

conversion_table = pd.DataFrame({
    "Logit": example_logits.numpy(),
    "Sigmoid Probability": example_probabilities.numpy()
})

display(conversion_table)

print("""
INTERPRETATION:

Negative logit
→ Low probability of positive relationship

Zero logit
→ 50% probability

Positive logit
→ High probability of positive relationship
""")

print("=" * 100)

LOGIT TO PROBABILITY CONVERSION


,Logit,Sigmoid Probability
0,-5.0,0.006693
1,-2.0,0.119203
2,0.0,0.500000
3,2.0,0.880797
4,5.0,0.993307



INTERPRETATION:

Negative logit
→ Low probability of positive relationship

Zero logit
→ 50% probability

Positive logit
→ High probability of positive relationship



In [13]:
# ==================================================================================================
# CELL 9 — DEFINE BINARY CROSS-ENTROPY OBJECTIVE
# ==================================================================================================

print("=" * 100)
print("BINARY CROSS-ENTROPY LOSS OBJECTIVE")
print("=" * 100)

print(r"""
Binary Cross Entropy:

        L = -[ y log(p) + (1-y) log(1-p) ]

Where:

    y = true relationship label
    p = predicted probability

For a positive pair:

    y = 1

    L = -log(p)

Therefore:

    If p → 1
        Loss → 0

    If p → 0
        Loss → very large


For a negative pair:

    y = 0

    L = -log(1-p)

Therefore:

    If p → 0
        Loss → 0

    If p → 1
        Loss → very large


Therefore the loss penalizes confident incorrect predictions.
""")

print("=" * 100)

BINARY CROSS-ENTROPY LOSS OBJECTIVE

Binary Cross Entropy:

        L = -[ y log(p) + (1-y) log(1-p) ]

Where:

    y = true relationship label
    p = predicted probability

For a positive pair:

    y = 1

    L = -log(p)

Therefore:

    If p → 1
        Loss → 0

    If p → 0
        Loss → very large


For a negative pair:

    y = 0

    L = -log(1-p)

Therefore:

    If p → 0
        Loss → 0

    If p → 1
        Loss → very large


Therefore the loss penalizes confident incorrect predictions.



In [14]:
# ==================================================================================================
# CELL 10 — DEMONSTRATE CROSS-MODAL LOSS BEHAVIOUR
# ==================================================================================================

print("=" * 100)
print("DEMONSTRATING LOSS BEHAVIOUR")
print("=" * 100)

loss_function_demo = nn.BCEWithLogitsLoss()

# Example logits
demo_logits = torch.tensor([
    4.0,    # confidently positive
    -4.0,   # confidently negative
    0.0,    # uncertain
    -4.0    # confidently wrong for positive
])

# Corresponding true labels
demo_labels = torch.tensor([
    1.0,
    0.0,
    1.0,
    1.0
])

individual_loss_function = nn.BCEWithLogitsLoss(
    reduction="none"
)

individual_losses = individual_loss_function(
    demo_logits,
    demo_labels
)

demo_probabilities = torch.sigmoid(demo_logits)

loss_demo_table = pd.DataFrame({
    "Logit": demo_logits.numpy(),
    "True_Label": demo_labels.numpy(),
    "Probability": demo_probabilities.numpy(),
    "Loss": individual_losses.detach().numpy()
})

display(loss_demo_table)

print("""
KEY OBSERVATION:

Correct confident prediction
→ Small loss

Uncertain prediction
→ Moderate loss

Confident incorrect prediction
→ Very large loss

This behaviour provides the learning signal for the neural network.
""")

print("=" * 100)

DEMONSTRATING LOSS BEHAVIOUR


,Logit,True_Label,Probability,Loss
0,4.0,1.0,0.982014,0.018150
1,-4.0,0.0,0.017986,0.018150
2,0.0,1.0,0.500000,0.693147
3,-4.0,1.0,0.017986,4.018150



KEY OBSERVATION:

Correct confident prediction
→ Small loss

Uncertain prediction
→ Moderate loss

Confident incorrect prediction
→ Very large loss

This behaviour provides the learning signal for the neural network.



In [15]:
# ==================================================================================================
# CELL 11 — DEFINE BASELINE CROSS-MODAL LOSS FUNCTION
# ==================================================================================================

print("=" * 100)
print("DEFINING BASELINE CROSS-MODAL LOSS")
print("=" * 100)

baseline_loss_function = nn.BCEWithLogitsLoss()

print("""
Selected Loss Function:

    BCEWithLogitsLoss

Mathematical components:

    Raw Model Logit
            ↓
    Numerically Stable Sigmoid
            +
    Binary Cross Entropy
            ↓
        Final Loss

Why selected?

✓ Binary pair labels (0 / 1)
✓ Direct compatibility with model logits
✓ Numerically stable
✓ Standard objective for binary relationship prediction
✓ No manual sigmoid required during training
""")

print("=" * 100)
print("✓ BASELINE LOSS FUNCTION DEFINED")
print("=" * 100)

DEFINING BASELINE CROSS-MODAL LOSS

Selected Loss Function:

    BCEWithLogitsLoss

Mathematical components:

    Raw Model Logit
            ↓
    Numerically Stable Sigmoid
            +
    Binary Cross Entropy
            ↓
        Final Loss

Why selected?

✓ Binary pair labels (0 / 1)
✓ Direct compatibility with model logits
✓ Numerically stable
✓ Standard objective for binary relationship prediction
✓ No manual sigmoid required during training

✓ BASELINE LOSS FUNCTION DEFINED


In [16]:
# ==================================================================================================
# CELL 12 — VERIFY LOSS FUNCTION WITH REAL CROSS-MODAL LABELS
# ==================================================================================================

print("=" * 100)
print("VERIFYING LOSS FUNCTION WITH REAL DATASET LABELS")
print("=" * 100)

# Select a small batch of real labels
sample_size = 32

sample_labels = torch.tensor(
    y_train[:sample_size],
    dtype=torch.float32
)

# Create example logits for loss verification
test_logits = torch.randn(
    sample_size,
    dtype=torch.float32
)

test_loss = baseline_loss_function(
    test_logits,
    sample_labels
)

print(f"\nSample size      : {sample_size}")
print(f"Logit shape      : {test_logits.shape}")
print(f"Label shape      : {sample_labels.shape}")
print(f"Computed loss    : {test_loss.item():.6f}")

assert torch.isfinite(test_loss), "Loss is NaN or Infinite."

print("\n✓ LOSS COMPUTED SUCCESSFULLY")
print("✓ LOSS IS FINITE")
print("✓ LOGIT / LABEL DIMENSIONS COMPATIBLE")

print("=" * 100)

VERIFYING LOSS FUNCTION WITH REAL DATASET LABELS

Sample size      : 32
Logit shape      : torch.Size([32])
Label shape      : torch.Size([32])
Computed loss    : 0.719950

✓ LOSS COMPUTED SUCCESSFULLY
✓ LOSS IS FINITE
✓ LOGIT / LABEL DIMENSIONS COMPATIBLE


In [17]:
# ==================================================================================================
# CELL 13 — DEFINE COMPLETE BASELINE LEARNING OBJECTIVE
# ==================================================================================================

print("=" * 100)
print("COMPLETE BASELINE CROSS-MODAL LEARNING OBJECTIVE")
print("=" * 100)

print(r"""
STEP 1 — INPUT PAIR

    (S, T)

    S = Source embedding (32D)
    T = Target embedding (32D)


STEP 2 — FEATURE FUSION

    Z = [S, T, |S-T|, S×T]

    Dimension:

    32 + 32 + 32 + 32 = 128


STEP 3 — BASELINE MODEL

    z = f(Z)

    Output:

    One raw relationship logit


STEP 4 — RELATIONSHIP LOSS

    L = BCEWithLogitsLoss(z, y)


STEP 5 — OPTIMIZATION GOAL

    Minimize:

    min L


Therefore:

Positive relationship (y = 1)
→ Model learns higher logits

Negative relationship (y = 0)
→ Model learns lower logits
""")

print("=" * 100)

COMPLETE BASELINE CROSS-MODAL LEARNING OBJECTIVE

STEP 1 — INPUT PAIR

    (S, T)

    S = Source embedding (32D)
    T = Target embedding (32D)


STEP 2 — FEATURE FUSION

    Z = [S, T, |S-T|, S×T]

    Dimension:

    32 + 32 + 32 + 32 = 128


STEP 3 — BASELINE MODEL

    z = f(Z)

    Output:

    One raw relationship logit


STEP 4 — RELATIONSHIP LOSS

    L = BCEWithLogitsLoss(z, y)


STEP 5 — OPTIMIZATION GOAL

    Minimize:

    min L


Therefore:

Positive relationship (y = 1)
→ Model learns higher logits

Negative relationship (y = 0)
→ Model learns lower logits



In [18]:
# ==================================================================================================
# CELL 14 — BASELINE LOSS DESIGN JUSTIFICATION
# ==================================================================================================

print("=" * 100)
print("BASELINE LOSS DESIGN JUSTIFICATION")
print("=" * 100)

justification = {
    "Dataset Property": [
        "Relationship labels",
        "Model output",
        "Classification objective",
        "Class balance",
        "Numerical stability",
        "Baseline simplicity"
    ],

    "Observation": [
        "Binary: Positive (1) / Negative (0)",
        "Single raw relationship logit",
        "Binary relationship prediction",
        "Balanced positive and negative pairs",
        "BCEWithLogitsLoss is numerically stable",
        "Simple objective required before GAN comparison"
    ],

    "Loss Design Decision": [
        "Binary Cross Entropy",
        "Use BCEWithLogitsLoss",
        "Predict relationship probability",
        "No class weighting initially required",
        "Avoid separate sigmoid during training",
        "No adversarial loss in baseline"
    ]
}

justification_df = pd.DataFrame(justification)

display(justification_df)

print("\n" + "=" * 100)
print("✓ BASELINE LOSS DESIGN JUSTIFIED")
print("=" * 100)

BASELINE LOSS DESIGN JUSTIFICATION


,Dataset Property,Observation,Loss Design Decision
0,Relationship labels,Binary: Positive (1) / Negative (0),Binary Cross Entropy
1,Model output,Single raw relationship logit,Use BCEWithLogitsLoss
2,Classification objective,Binary relationship prediction,Predict relationship probability
3,Class balance,Balanced positive and negative pairs,No class weighting initially required
4,Numerical stability,BCEWithLogitsLoss is numerically stable,Avoid separate sigmoid during training
5,Baseline simplicity,Simple objective required before GAN comparison,No adversarial loss in baseline



✓ BASELINE LOSS DESIGN JUSTIFIED


In [19]:
# ==================================================================================================
# CELL 15 — COMPARE CORRECT AND INCORRECT PREDICTION LOSSES
# ==================================================================================================

print("=" * 100)
print("CORRECT VS INCORRECT PREDICTION LOSS ANALYSIS")
print("=" * 100)

comparison_logits = torch.tensor([
    5.0,    # correct confident positive
    -5.0,   # correct confident negative
    -5.0,   # incorrect confident prediction
    5.0     # incorrect confident prediction
])

comparison_labels = torch.tensor([
    1.0,
    0.0,
    1.0,
    0.0
])

comparison_loss_function = nn.BCEWithLogitsLoss(
    reduction="none"
)

comparison_losses = comparison_loss_function(
    comparison_logits,
    comparison_labels
)

comparison_probabilities = torch.sigmoid(
    comparison_logits
)

comparison_df = pd.DataFrame({
    "Logit": comparison_logits.numpy(),
    "True Label": comparison_labels.numpy(),
    "Probability": comparison_probabilities.numpy(),
    "Loss": comparison_losses.numpy()
})

display(comparison_df)

print("""
EXPECTED BEHAVIOUR:

Correct + Confident
→ Very small loss

Incorrect + Confident
→ Very large loss

This is exactly the behaviour required for relationship learning.
""")

print("=" * 100)

CORRECT VS INCORRECT PREDICTION LOSS ANALYSIS


,Logit,True Label,Probability,Loss
0,5.0,1.0,0.993307,0.006715
1,-5.0,0.0,0.006693,0.006715
2,-5.0,1.0,0.006693,5.006715
3,5.0,0.0,0.993307,5.006715



EXPECTED BEHAVIOUR:

Correct + Confident
→ Very small loss

Incorrect + Confident
→ Very large loss

This is exactly the behaviour required for relationship learning.



In [20]:
# ==================================================================================================
# CELL 16 — DEFINE BASELINE LOSS CONFIGURATION
# ==================================================================================================

LOSS_CONFIG = {
    "loss_name": "BCEWithLogitsLoss",
    "task_type": "binary_cross_modal_relationship_classification",
    "positive_label": 1,
    "negative_label": 0,
    "model_output": "raw_relationship_logit",
    "sigmoid_applied_in_loss": True,
    "class_weighting": False,
    "reduction": "mean"
}

print("=" * 100)
print("BASELINE LOSS CONFIGURATION")
print("=" * 100)

for key, value in LOSS_CONFIG.items():
    print(f"{key:<35}: {value}")

print("\n" + "=" * 100)
print("✓ LOSS CONFIGURATION DEFINED")
print("=" * 100)

BASELINE LOSS CONFIGURATION
loss_name                          : BCEWithLogitsLoss
task_type                          : binary_cross_modal_relationship_classification
positive_label                     : 1
negative_label                     : 0
model_output                       : raw_relationship_logit
sigmoid_applied_in_loss            : True
class_weighting                    : False
reduction                          : mean

✓ LOSS CONFIGURATION DEFINED


In [21]:
# ==================================================================================================
# CELL 17 — SAVE CROSS-MODAL LOSS CONFIGURATION
# ==================================================================================================

import json

OUTPUT_DIR = (
    PROJECT_ROOT /
    "06_cross_modal_learning" /
    "03_cross_modal_learning" /
    "03_03_cross_modal_loss_design"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOSS_CONFIG_PATH = (
    OUTPUT_DIR /
    "03_03_baseline_loss_configuration.json"
)

with open(
    LOSS_CONFIG_PATH,
    "w"
) as file:

    json.dump(
        LOSS_CONFIG,
        file,
        indent=4
    )

print("=" * 100)
print("SAVING BASELINE LOSS CONFIGURATION")
print("=" * 100)

print(f"\nSaved path:\n{LOSS_CONFIG_PATH}")
print(f"\nFile exists: {LOSS_CONFIG_PATH.exists()}")

assert LOSS_CONFIG_PATH.exists()

print("\n✓ LOSS CONFIGURATION SAVED SUCCESSFULLY")
print("=" * 100)

SAVING BASELINE LOSS CONFIGURATION

Saved path:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_03_cross_modal_loss_design\03_03_baseline_loss_configuration.json

File exists: True

✓ LOSS CONFIGURATION SAVED SUCCESSFULLY


In [22]:
# ==================================================================================================
# CELL 18 — FINAL LOSS DESIGN VERIFICATION
# ==================================================================================================

print("=" * 100)
print("FINAL CROSS-MODAL LOSS DESIGN VERIFICATION")
print("=" * 100)

checks = {
    "Binary relationship labels": (
        set(np.unique(y_train)).issubset({0, 1})
        and set(np.unique(y_val)).issubset({0, 1})
    ),

    "Training label balance": (
        np.sum(y_train == 1) == np.sum(y_train == 0)
    ),

    "Validation label balance": (
        np.sum(y_val == 1) == np.sum(y_val == 0)
    ),

    "Loss function defined": (
        baseline_loss_function is not None
    ),

    "Loss numerical stability": (
        torch.isfinite(test_loss).item()
    ),

    "Loss configuration saved": (
        LOSS_CONFIG_PATH.exists()
    )
}

all_passed = True

print()

for check_name, status in checks.items():

    symbol = "✓" if status else "✗"

    print(
        f"{symbol} {check_name}"
    )

    if not status:
        all_passed = False


print("\n" + "=" * 100)

if all_passed:

    print("FINAL STATUS: LOSS DESIGN VERIFIED")
    print("\n✓ READY FOR BASELINE TRAINING")

else:

    print("FINAL STATUS: VERIFICATION FAILED")
    print("\nResolve failed checks before training.")

print("=" * 100)

assert all_passed, "Loss design verification failed."

FINAL CROSS-MODAL LOSS DESIGN VERIFICATION

✓ Binary relationship labels
✓ Training label balance
✓ Validation label balance
✓ Loss function defined
✓ Loss numerical stability
✓ Loss configuration saved

FINAL STATUS: LOSS DESIGN VERIFIED

✓ READY FOR BASELINE TRAINING


In [23]:
# ==================================================================================================
# CELL 19 — CROSS-MODAL LOSS DESIGN SUMMARY
# ==================================================================================================

print("=" * 100)
print("PHASE 03.03 — CROSS-MODAL LOSS DESIGN SUMMARY")
print("=" * 100)

summary = pd.DataFrame({
    "Component": [
        "Learning Task",
        "Input",
        "Model Output",
        "Positive Label",
        "Negative Label",
        "Probability Function",
        "Loss Function",
        "Training Objective",
        "Class Weighting",
        "Loss Configuration"
    ],

    "Design": [
        "Binary Cross-Modal Relationship Classification",
        "Source (32D) + Target (32D)",
        "Single Raw Relationship Logit",
        "1",
        "0",
        "Sigmoid",
        "BCEWithLogitsLoss",
        "Minimize Binary Cross-Entropy",
        "Not Required — Balanced Classes",
        "Saved as JSON"
    ]
})

display(summary)

print("\n" + "=" * 100)
print("✓ PHASE 03.03 SUMMARY CREATED")
print("=" * 100)

PHASE 03.03 — CROSS-MODAL LOSS DESIGN SUMMARY


,Component,Design
0,Learning Task,Binary Cross-Modal Relationship Classification
1,Input,Source (32D) + Target (32D)
2,Model Output,Single Raw Relationship Logit
3,Positive Label,1
4,Negative Label,0
5,Probability Function,Sigmoid
6,Loss Function,BCEWithLogitsLoss
7,Training Objective,Minimize Binary Cross-Entropy
8,Class Weighting,Not Required — Balanced Classes
9,Loss Configuration,Saved as JSON



✓ PHASE 03.03 SUMMARY CREATED


In [24]:
# ==================================================================================================
# CONCLUSION — PHASE 03.03 CROSS-MODAL LOSS DESIGN
# ==================================================================================================

print("=" * 100)
print("CONCLUSION — PHASE 03.03 CROSS-MODAL LOSS DESIGN")
print("=" * 100)

print("""
The mathematical learning objective for the baseline cross-modal model
was successfully defined.

The cross-modal dataset contains binary semantic relationships:

    Positive relationship → Label 1
    Negative relationship → Label 0

The baseline model produces one raw relationship logit for every
source-target representation pair.

The learning objective uses:

    BCEWithLogitsLoss

which combines sigmoid activation and Binary Cross Entropy in a
numerically stable formulation.

The mathematical objective is:

    L = -[y log(p) + (1-y) log(1-p)]

This objective encourages:

    Positive pairs
    → Higher relationship probabilities

    Negative pairs
    → Lower relationship probabilities

The baseline loss intentionally remains simple.

No adversarial, reconstruction, or GAN-specific losses were introduced
at this stage because the objective is to establish a scientifically
fair conventional neural network baseline.

The loss function was:

✓ Mathematically defined
✓ Tested with example predictions
✓ Verified with real dataset labels
✓ Checked for numerical stability
✓ Configured for reproducibility
✓ Saved for experiment documentation

The baseline model architecture and learning objective are now complete.

NEXT PHASE:

03.04 — BASELINE CROSS-MODAL TRAINING

The baseline model will be trained using the representation-aware
training dataset and evaluated on the independent validation dataset.

This will establish the minimum performance benchmark before introducing
GAN-based adversarial learning.

""")

print("=" * 100)
print("✓ PHASE 03.03 COMPLETED")
print("=" * 100)

CONCLUSION — PHASE 03.03 CROSS-MODAL LOSS DESIGN

The mathematical learning objective for the baseline cross-modal model
was successfully defined.

The cross-modal dataset contains binary semantic relationships:

    Positive relationship → Label 1
    Negative relationship → Label 0

The baseline model produces one raw relationship logit for every
source-target representation pair.

The learning objective uses:

    BCEWithLogitsLoss

which combines sigmoid activation and Binary Cross Entropy in a
numerically stable formulation.

The mathematical objective is:

    L = -[y log(p) + (1-y) log(1-p)]

This objective encourages:

    Positive pairs
    → Higher relationship probabilities

    Negative pairs
    → Lower relationship probabilities

The baseline loss intentionally remains simple.

No adversarial, reconstruction, or GAN-specific losses were introduced
at this stage because the objective is to establish a scientifically
fair conventional neural network baseline.

The loss func